In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
# Load the trained deep learning model
model  = load_model('../model_artifacts/model.keras') 

# Load the Gender label encoder
with open('../model_artifacts/label_encoder_gender.pkl', mode='rb') as file:
    label_encoder_gender = pickle.load(file)

# Load the Geography one-hot encoder
with open('../model_artifacts/onehot_encoder_geo.pkl', mode='rb') as file:
    onehot_encoder_geo = pickle.load(file)   
    
# Load the feature scaler
with open('../model_artifacts/scaler.pkl', mode='rb') as file:
    scaler = pickle.load(file)    

In [3]:
type(onehot_encoder_geo)

sklearn.preprocessing._encoders.OneHotEncoder

In [4]:
# Example input data 
# Example input data
input_data = {
    'CreditScore': 650,
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 70000,
    'Geography': 'France'
}

In [5]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

d:\ML_DL_NLP_Bootcamp\Customer-Churn-Deep-Learning\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
# Convert the customer input dictionary into a DataFrame
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography
0,650,Male,40,3,60000,2,1,1,70000,France


In [11]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography
0,650,1,40,3,60000,2,1,1,70000,France


In [ ]:
# Remove the original Geography column and add the one-hot encoded Geography columns
df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
df



,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,650,1,40,3,60000,2,1,1,70000,1.0,0.0,0.0


In [ ]:
# Scale the input data using the same scaler used during training
input_scaled = scaler.transform(df)
input_scaled

array([[-0.01709861,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.5290988 ,  1.00150113,
        -0.57946723, -0.57638802]])

In [ ]:
# Predict the probability of customer churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


array([[0.02821164]], dtype=float32)

In [17]:
prediction_probability  = prediction[0][0]

In [18]:
prediction_probability

np.float32(0.028211636)

In [19]:
if prediction_probability > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
